# 105 — TCGA Methylation Cohort Freeze

## Objective

Freeze the exact source-complete TCGA primary-tumor methylation cohort selected for acquisition after the platform-coverage assessment performed in notebook 104.

This notebook converts the all-platform GDC query exports into a version-controlled HM27/HM450 file-level cohort that can be downloaded reproducibly with the GDC Data Transfer Tool.

## Input candidate cohort

The candidate all-platform query is represented by:

- `config/manifests/tcga_methylation/gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values.txt`
- `config/manifests/tcga_methylation/gdc_sample_sheet_tcga_primary_tumor_methylation_array_sesame_beta_values.tsv`
- `config/manifests/tcga_methylation/gdc_metadata_tcga_primary_tumor_methylation_array_sesame_beta_values.json`

These files contain harmonized SeSAMe methylation beta-value records from HM27, HM450, and EPIC v2 arrays.

## Platform-selection decision

Notebook 104 established the following acquisition strategy:

- retain all HM27 files;
- retain all HM450 files;
- exclude EPIC v2 files;
- retain the complete methylation source cohort rather than restricting acquisition to cases currently represented in the RNA-seq cohort;
- defer sample, aliquot, and duplicate-file selection until methylation quality control.

The source-complete HM27/HM450 cohort is expected to contain:

- 10,897 files;
- 10,703 aliquots;
- 10,656 samples;
- 10,610 cases;
- 33 TCGA projects;
- 106.736 GiB of payload data.

HM27 is retained because it provides substantial additional coverage in several projects, including TCGA-OV and TCGA-GBM. EPIC v2 is excluded because its 53 files contribute no exclusive cases and would introduce an additional platform without increasing cohort coverage.

## Outputs

This notebook will write the following version-controlled cohort-freeze artifacts:

- `config/manifests/tcga_methylation/gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.txt`
- `config/manifests/tcga_methylation/gdc_sample_sheet_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv`
- `config/manifests/tcga_methylation/gdc_file_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv`
- `config/manifests/tcga_methylation/gdc_cohort_freeze_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.json`

The selected manifest will serve as the authoritative input for the GDC Data Transfer Tool.

## Scope

This notebook performs file-level cohort selection and provenance freezing only. It does not:

- download methylation payloads;
- parse beta-value files;
- harmonize HM27 and HM450 probes;
- pool platforms;
- select one sample or aliquot per case;
- apply methylation quality-control exclusions;
- construct the final RNA–methylation cohort;
- perform biological analysis.

Raw candidate exports remain immutable. Platform-aware processing and sample-level decisions will be handled explicitly in downstream notebooks.

In [1]:
# =============================================================================
# 105 — TCGA Methylation Cohort Freeze
# Imports, project-root detection, and file paths
# =============================================================================

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


# -----------------------------------------------------------------------------
# Detect project root
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "pancancer-epigenetics":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not locate the pancancer-epigenetics project root."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# -----------------------------------------------------------------------------
# Project utilities
# -----------------------------------------------------------------------------

from src.utils.paths import Paths
from src.utils.file_checks import (
    calculate_sha256,
    get_file_size_mb,
    to_project_relative_posix_path,
)


# -----------------------------------------------------------------------------
# Source files: immutable all-platform GDC exports
# -----------------------------------------------------------------------------

METHYLATION_MANIFEST_DIR = (
    Paths.config / "manifests" / "tcga_methylation"
)

SOURCE_MANIFEST_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values.txt"
)

SOURCE_SAMPLE_SHEET_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_sample_sheet_tcga_primary_tumor_methylation_array_sesame_beta_values.tsv"
)

SOURCE_METADATA_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_metadata_tcga_primary_tumor_methylation_array_sesame_beta_values.json"
)


# -----------------------------------------------------------------------------
# Frozen HM27 + HM450 cohort outputs
# -----------------------------------------------------------------------------

FROZEN_MANIFEST_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.txt"
)

FROZEN_SAMPLE_SHEET_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_sample_sheet_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv"
)

FROZEN_FILE_INDEX_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_file_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv"
)

FROZEN_COHORT_METADATA_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_cohort_freeze_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.json"
)


print(f"Project root:              {PROJECT_ROOT}")
print(f"Source manifest:           {SOURCE_MANIFEST_PATH}")
print(f"Source sample sheet:       {SOURCE_SAMPLE_SHEET_PATH}")
print(f"Source metadata:           {SOURCE_METADATA_PATH}")
print(f"Frozen manifest:           {FROZEN_MANIFEST_PATH}")
print(f"Frozen sample sheet:       {FROZEN_SAMPLE_SHEET_PATH}")
print(f"Frozen file index:         {FROZEN_FILE_INDEX_PATH}")
print(f"Frozen cohort metadata:    {FROZEN_COHORT_METADATA_PATH}")

Project root:              C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics
Source manifest:           C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values.txt
Source sample sheet:       C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_sample_sheet_tcga_primary_tumor_methylation_array_sesame_beta_values.tsv
Source metadata:           C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_metadata_tcga_primary_tumor_methylation_array_sesame_beta_values.json
Frozen manifest:           C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.txt
Frozen sample sheet:       C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epig

In [2]:
# =============================================================================
# Validate and load source GDC exports
# =============================================================================

source_input_paths = {
    "manifest": SOURCE_MANIFEST_PATH,
    "sample_sheet": SOURCE_SAMPLE_SHEET_PATH,
    "metadata": SOURCE_METADATA_PATH,
}

frozen_output_paths = {
    "manifest": FROZEN_MANIFEST_PATH,
    "sample_sheet": FROZEN_SAMPLE_SHEET_PATH,
    "file_index": FROZEN_FILE_INDEX_PATH,
    "cohort_metadata": FROZEN_COHORT_METADATA_PATH,
}


# -----------------------------------------------------------------------------
# Validate source paths
# -----------------------------------------------------------------------------

missing_inputs = [
    path
    for path in source_input_paths.values()
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing required methylation source files:\n"
        + "\n".join(str(path) for path in missing_inputs)
    )

non_file_inputs = [
    path
    for path in source_input_paths.values()
    if not path.is_file()
]

if non_file_inputs:
    raise ValueError(
        "The following source paths are not files:\n"
        + "\n".join(str(path) for path in non_file_inputs)
    )


# Ensure that no output path can overwrite a source export.
source_paths_resolved = {
    path.resolve()
    for path in source_input_paths.values()
}

output_paths_resolved = {
    path.resolve()
    for path in frozen_output_paths.values()
}

overlapping_paths = source_paths_resolved & output_paths_resolved

if overlapping_paths:
    raise ValueError(
        "A frozen output path overlaps a source input path:\n"
        + "\n".join(str(path) for path in sorted(overlapping_paths))
    )

if len(output_paths_resolved) != len(frozen_output_paths):
    raise ValueError("Frozen output paths are not unique.")


# -----------------------------------------------------------------------------
# Load tabular exports
# -----------------------------------------------------------------------------

source_manifest = pd.read_csv(
    SOURCE_MANIFEST_PATH,
    sep="\t",
    dtype={
        "id": "string",
        "filename": "string",
        "md5": "string",
        "state": "string",
    },
)

source_sample_sheet = pd.read_csv(
    SOURCE_SAMPLE_SHEET_PATH,
    sep="\t",
    dtype="string",
)

source_manifest["size"] = pd.to_numeric(
    source_manifest["size"],
    errors="raise",
).astype("int64")


# -----------------------------------------------------------------------------
# Load metadata export
# -----------------------------------------------------------------------------

with SOURCE_METADATA_PATH.open("r", encoding="utf-8") as handle:
    source_metadata_records = json.load(handle)

if not isinstance(source_metadata_records, list):
    raise TypeError(
        "Expected the GDC metadata export to contain a top-level list."
    )

if not source_metadata_records:
    raise ValueError("The GDC metadata export contains no records.")


# -----------------------------------------------------------------------------
# Loading summary
# -----------------------------------------------------------------------------

print("Source GDC exports validated and loaded.")

for label, path in source_input_paths.items():
    size_mib = path.stat().st_size / (1024**2)
    print(f"- {label}: {size_mib:,.3f} MiB")

print()
print(f"Manifest shape:          {source_manifest.shape}")
print(f"Sample-sheet shape:      {source_sample_sheet.shape}")
print(f"Metadata records:        {len(source_metadata_records):,}")
print(f"Existing frozen outputs: {sum(path.exists() for path in frozen_output_paths.values())}")


Source GDC exports validated and loaded.
- manifest: 1.729 MiB
- sample_sheet: 2.359 MiB
- metadata: 14.067 MiB

Manifest shape:          (10950, 5)
Sample-sheet shape:      (10950, 11)
Metadata records:        10,950
Existing frozen outputs: 0


In [3]:
# =============================================================================
# Normalize and validate source GDC metadata
# =============================================================================

PLATFORM_HM27 = "Illumina Human Methylation 27"
PLATFORM_HM450 = "Illumina Human Methylation 450"
PLATFORM_EPICV2 = "Illumina Methylation Epic v2"

INCLUDED_PLATFORMS = frozenset({
    PLATFORM_HM27,
    PLATFORM_HM450,
})

EXPECTED_SOURCE_PLATFORMS = {
    PLATFORM_HM27,
    PLATFORM_HM450,
    PLATFORM_EPICV2,
}


# -----------------------------------------------------------------------------
# Validate required source columns
# -----------------------------------------------------------------------------

required_manifest_columns = {
    "id",
    "filename",
    "md5",
    "size",
    "state",
}

required_sample_sheet_columns = {
    "File ID",
    "File Name",
    "Data Category",
    "Data Type",
    "Project ID",
    "Case ID",
    "Sample ID",
    "Tissue Type",
    "Tumor Descriptor",
    "Specimen Type",
    "Preservation Method",
}

missing_manifest_columns = (
    required_manifest_columns - set(source_manifest.columns)
)

missing_sample_sheet_columns = (
    required_sample_sheet_columns - set(source_sample_sheet.columns)
)

if missing_manifest_columns:
    raise KeyError(
        "Missing manifest columns: "
        + ", ".join(sorted(missing_manifest_columns))
    )

if missing_sample_sheet_columns:
    raise KeyError(
        "Missing sample-sheet columns: "
        + ", ".join(sorted(missing_sample_sheet_columns))
    )


# -----------------------------------------------------------------------------
# Normalize file-level metadata and associated entities
# -----------------------------------------------------------------------------

metadata_rows = []
association_rows = []

for record in source_metadata_records:
    analysis = record.get("analysis") or {}
    associated_entities = record.get("associated_entities") or []

    if not isinstance(associated_entities, list):
        raise TypeError(
            f"associated_entities is not a list for file "
            f"{record.get('file_id')}"
        )

    metadata_rows.append(
        {
            "file_id": record.get("file_id"),
            "file_name": record.get("file_name"),
            "file_submitter_id": record.get("submitter_id"),
            "md5": record.get("md5sum"),
            "file_size_bytes": record.get("file_size"),
            "state": record.get("state"),
            "data_format": record.get("data_format"),
            "access": record.get("access"),
            "data_category": record.get("data_category"),
            "data_type": record.get("data_type"),
            "experimental_strategy": record.get("experimental_strategy"),
            "platform": record.get("platform"),
            "workflow_type": analysis.get("workflow_type"),
        }
    )

    for entity in associated_entities:
        association_rows.append(
            {
                "file_id": record.get("file_id"),
                "case_uuid": entity.get("case_id"),
                "entity_type": entity.get("entity_type"),
                "entity_uuid": entity.get("entity_id"),
                "entity_submitter_id": entity.get(
                    "entity_submitter_id"
                ),
            }
        )

metadata_file_index = pd.DataFrame(metadata_rows)
metadata_associations = pd.DataFrame(association_rows)

metadata_file_index["file_size_bytes"] = pd.to_numeric(
    metadata_file_index["file_size_bytes"],
    errors="raise",
).astype("int64")


# -----------------------------------------------------------------------------
# Fail-closed metadata validation
# -----------------------------------------------------------------------------

required_metadata_fields = [
    "file_id",
    "file_name",
    "md5",
    "file_size_bytes",
    "state",
    "data_format",
    "access",
    "data_category",
    "data_type",
    "experimental_strategy",
    "platform",
    "workflow_type",
]

null_metadata_fields = {
    column: int(metadata_file_index[column].isna().sum())
    for column in required_metadata_fields
    if metadata_file_index[column].isna().any()
}

if null_metadata_fields:
    raise ValueError(
        f"Required metadata fields contain null values: "
        f"{null_metadata_fields}"
    )

if metadata_file_index["file_id"].duplicated().any():
    raise ValueError("Duplicate file IDs found in metadata.")

if source_manifest["id"].duplicated().any():
    raise ValueError("Duplicate file IDs found in source manifest.")

if source_sample_sheet["File ID"].duplicated().any():
    raise ValueError("Duplicate file IDs found in source sample sheet.")


# Every file should have exactly one aliquot association.
association_counts = (
    metadata_associations
    .groupby("file_id")
    .size()
    .reindex(metadata_file_index["file_id"], fill_value=0)
)

if not association_counts.eq(1).all():
    raise ValueError(
        "Expected exactly one associated entity per methylation file."
    )

observed_entity_types = set(
    metadata_associations["entity_type"].dropna()
)

if observed_entity_types != {"aliquot"}:
    raise ValueError(
        f"Unexpected associated entity types: "
        f"{sorted(observed_entity_types)}"
    )


# -----------------------------------------------------------------------------
# Validate file-ID agreement across all three exports
# -----------------------------------------------------------------------------

manifest_file_ids = set(source_manifest["id"])
sample_sheet_file_ids = set(source_sample_sheet["File ID"])
metadata_file_ids = set(metadata_file_index["file_id"])

file_id_differences = {
    "manifest_only_vs_sample_sheet":
        manifest_file_ids - sample_sheet_file_ids,
    "sample_sheet_only_vs_manifest":
        sample_sheet_file_ids - manifest_file_ids,
    "manifest_only_vs_metadata":
        manifest_file_ids - metadata_file_ids,
    "metadata_only_vs_manifest":
        metadata_file_ids - manifest_file_ids,
}

nonempty_file_id_differences = {
    label: values
    for label, values in file_id_differences.items()
    if values
}

if nonempty_file_id_differences:
    raise ValueError(
        "File-ID disagreement across source exports: "
        + str({
            label: len(values)
            for label, values in nonempty_file_id_differences.items()
        })
    )


# -----------------------------------------------------------------------------
# Validate source query definition
# -----------------------------------------------------------------------------

expected_single_values = {
    "state": "released",
    "data_format": "TXT",
    "access": "open",
    "data_category": "DNA Methylation",
    "data_type": "Methylation Beta Value",
    "experimental_strategy": "Methylation Array",
    "workflow_type": "SeSAMe Methylation Beta Estimation",
}

for column, expected_value in expected_single_values.items():
    observed_values = set(
        metadata_file_index[column].dropna().unique()
    )

    if observed_values != {expected_value}:
        raise ValueError(
            f"Unexpected values in {column}: "
            f"{sorted(observed_values)}"
        )

observed_platforms = set(
    metadata_file_index["platform"].dropna().unique()
)

if observed_platforms != EXPECTED_SOURCE_PLATFORMS:
    raise ValueError(
        f"Unexpected methylation platforms: "
        f"{sorted(observed_platforms)}"
    )

if set(source_sample_sheet["Tissue Type"].dropna().unique()) != {"Tumor"}:
    raise ValueError("Source sample sheet contains non-tumor tissue.")

if set(
    source_sample_sheet["Tumor Descriptor"].dropna().unique()
) != {"Primary"}:
    raise ValueError(
        "Source sample sheet contains non-primary tumors."
    )


# -----------------------------------------------------------------------------
# Source platform summary
# -----------------------------------------------------------------------------

source_platform_summary = (
    metadata_file_index
    .groupby("platform", as_index=False)
    .agg(
        file_count=("file_id", "nunique"),
        total_size_bytes=("file_size_bytes", "sum"),
    )
)

source_platform_summary["total_size_gib"] = (
    source_platform_summary["total_size_bytes"] / (1024**3)
)

source_platform_summary = source_platform_summary.sort_values(
    "file_count",
    ascending=False,
).reset_index(drop=True)

print("Source metadata normalized and validated.")
print(f"File records:        {len(metadata_file_index):,}")
print(f"Entity associations: {len(metadata_associations):,}")
print(f"Platforms:           {len(observed_platforms)}")
print()
print(source_platform_summary)

Source metadata normalized and validated.
File records:        10,950
Entity associations: 10,950
Platforms:           3

                         platform  file_count  total_size_bytes  \
0  Illumina Human Methylation 450        8618      112853901884   
1   Illumina Human Methylation 27        2279        1752659375   
2    Illumina Methylation Epic v2          53        1351253876   

   total_size_gib  
0      105.103386  
1        1.632291  
2        1.258453  


In [4]:
# =============================================================================
# Build and validate the frozen HM27 + HM450 cohort in memory
# =============================================================================

# -----------------------------------------------------------------------------
# Add analysis-level identifiers to the normalized metadata
# -----------------------------------------------------------------------------

analysis_rows = []

for record in source_metadata_records:
    analysis = record.get("analysis") or {}

    analysis_rows.append(
        {
            "file_id": record.get("file_id"),
            "analysis_id": analysis.get("analysis_id"),
            "workflow_version": analysis.get("workflow_version"),
        }
    )

analysis_details = pd.DataFrame(analysis_rows)

if analysis_details["file_id"].duplicated().any():
    raise ValueError("Duplicate file IDs found in analysis metadata.")

if set(analysis_details["file_id"]) != metadata_file_ids:
    raise ValueError(
        "Analysis-level metadata does not match file-level metadata."
    )

metadata_file_index = metadata_file_index.merge(
    analysis_details,
    on="file_id",
    how="left",
    validate="one_to_one",
)


# -----------------------------------------------------------------------------
# Select the source-complete HM27 + HM450 cohort
# -----------------------------------------------------------------------------

selected_file_metadata = (
    metadata_file_index.loc[
        metadata_file_index["platform"].isin(INCLUDED_PLATFORMS)
    ]
    .copy()
)

selected_file_ids = set(selected_file_metadata["file_id"])

excluded_file_metadata = (
    metadata_file_index.loc[
        ~metadata_file_index["platform"].isin(INCLUDED_PLATFORMS)
    ]
    .copy()
)

frozen_manifest = (
    source_manifest.loc[
        source_manifest["id"].isin(selected_file_ids)
    ]
    .sort_values("id")
    .reset_index(drop=True)
    .copy()
)

frozen_sample_sheet = (
    source_sample_sheet.loc[
        source_sample_sheet["File ID"].isin(selected_file_ids)
    ]
    .sort_values("File ID")
    .reset_index(drop=True)
    .copy()
)

selected_associations = (
    metadata_associations.loc[
        metadata_associations["file_id"].isin(selected_file_ids)
    ]
    .rename(
        columns={
            "entity_uuid": "aliquot_uuid",
            "entity_submitter_id": "aliquot_submitter_id",
        }
    )
    .copy()
)


# -----------------------------------------------------------------------------
# Validate the platform selection
# -----------------------------------------------------------------------------

selected_platforms = set(
    selected_file_metadata["platform"].unique()
)

excluded_platforms = set(
    excluded_file_metadata["platform"].unique()
)

if selected_platforms != set(INCLUDED_PLATFORMS):
    raise ValueError(
        f"Unexpected selected platforms: {sorted(selected_platforms)}"
    )

if excluded_platforms != {PLATFORM_EPICV2}:
    raise ValueError(
        f"Unexpected excluded platforms: {sorted(excluded_platforms)}"
    )

if len(excluded_file_metadata) != 53:
    raise ValueError(
        "Expected exactly 53 excluded EPICv2 files, found "
        f"{len(excluded_file_metadata):,}."
    )

if len(selected_file_metadata) + len(excluded_file_metadata) != len(
    metadata_file_index
):
    raise ValueError(
        "Selected and excluded file sets do not reconstruct the source cohort."
    )

if set(frozen_manifest["id"]) != selected_file_ids:
    raise ValueError("Frozen manifest file-ID set is incomplete.")

if set(frozen_sample_sheet["File ID"]) != selected_file_ids:
    raise ValueError("Frozen sample-sheet file-ID set is incomplete.")

if set(selected_associations["file_id"]) != selected_file_ids:
    raise ValueError("Selected entity-association set is incomplete.")


# -----------------------------------------------------------------------------
# Validate manifest values against selected metadata
# -----------------------------------------------------------------------------

manifest_comparison = (
    frozen_manifest
    .rename(
        columns={
            "id": "file_id",
            "filename": "manifest_file_name",
            "md5": "manifest_md5",
            "size": "manifest_file_size_bytes",
            "state": "manifest_state",
        }
    )
    .merge(
        selected_file_metadata[
            [
                "file_id",
                "file_name",
                "md5",
                "file_size_bytes",
                "state",
            ]
        ],
        on="file_id",
        how="inner",
        validate="one_to_one",
    )
)

manifest_mismatches = {
    "file_name": int(
        manifest_comparison["manifest_file_name"]
        .ne(manifest_comparison["file_name"])
        .sum()
    ),
    "md5": int(
        manifest_comparison["manifest_md5"]
        .str.lower()
        .ne(manifest_comparison["md5"].str.lower())
        .sum()
    ),
    "file_size_bytes": int(
        manifest_comparison["manifest_file_size_bytes"]
        .ne(manifest_comparison["file_size_bytes"])
        .sum()
    ),
    "state": int(
        manifest_comparison["manifest_state"]
        .ne(manifest_comparison["state"])
        .sum()
    ),
}

if any(manifest_mismatches.values()):
    raise ValueError(
        f"Manifest-versus-metadata mismatches: "
        f"{manifest_mismatches}"
    )


# -----------------------------------------------------------------------------
# Normalize the selected sample sheet
# -----------------------------------------------------------------------------

sample_sheet_for_index = frozen_sample_sheet.rename(
    columns={
        "File ID": "file_id",
        "File Name": "sample_sheet_file_name",
        "Data Category": "sample_sheet_data_category",
        "Data Type": "sample_sheet_data_type",
        "Project ID": "project_id",
        "Case ID": "case_submitter_id",
        "Sample ID": "sample_submitter_id",
        "Tissue Type": "tissue_type",
        "Tumor Descriptor": "tumor_descriptor",
        "Specimen Type": "specimen_type",
        "Preservation Method": "preservation_method",
    }
)

selected_file_index = (
    selected_file_metadata
    .merge(
        selected_associations[
            [
                "file_id",
                "case_uuid",
                "aliquot_uuid",
                "aliquot_submitter_id",
            ]
        ],
        on="file_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        sample_sheet_for_index,
        on="file_id",
        how="left",
        validate="one_to_one",
    )
)

sample_sheet_mismatches = {
    "file_name": int(
        selected_file_index["sample_sheet_file_name"]
        .ne(selected_file_index["file_name"])
        .sum()
    ),
    "data_category": int(
        selected_file_index["sample_sheet_data_category"]
        .ne(selected_file_index["data_category"])
        .sum()
    ),
    "data_type": int(
        selected_file_index["sample_sheet_data_type"]
        .ne(selected_file_index["data_type"])
        .sum()
    ),
}

if any(sample_sheet_mismatches.values()):
    raise ValueError(
        f"Sample-sheet-versus-metadata mismatches: "
        f"{sample_sheet_mismatches}"
    )


# -----------------------------------------------------------------------------
# Build canonical 25-column file index
# -----------------------------------------------------------------------------

frozen_file_index_columns = [
    "file_id",
    "file_name",
    "file_submitter_id",
    "md5",
    "file_size_bytes",
    "state",
    "data_format",
    "access",
    "data_category",
    "data_type",
    "experimental_strategy",
    "platform",
    "workflow_type",
    "workflow_version",
    "analysis_id",
    "project_id",
    "case_uuid",
    "case_submitter_id",
    "sample_submitter_id",
    "aliquot_uuid",
    "aliquot_submitter_id",
    "tissue_type",
    "tumor_descriptor",
    "specimen_type",
    "preservation_method",
]

frozen_file_index = (
    selected_file_index[
        frozen_file_index_columns
    ]
    .sort_values("file_id")
    .reset_index(drop=True)
    .copy()
)

if frozen_file_index["file_id"].duplicated().any():
    raise ValueError("Duplicate file IDs found in frozen file index.")

if frozen_file_index[frozen_file_index_columns].isna().any().any():
    null_counts = (
        frozen_file_index[frozen_file_index_columns]
        .isna()
        .sum()
    )
    null_counts = null_counts[null_counts > 0].to_dict()

    raise ValueError(
        f"Frozen file index contains null identifiers or metadata: "
        f"{null_counts}"
    )


# -----------------------------------------------------------------------------
# Validate expected frozen-cohort counts
# -----------------------------------------------------------------------------

frozen_counts = {
    "files": frozen_file_index["file_id"].nunique(),
    "aliquots": frozen_file_index["aliquot_uuid"].nunique(),
    "samples": frozen_file_index["sample_submitter_id"].nunique(),
    "cases": frozen_file_index["case_uuid"].nunique(),
    "projects": frozen_file_index["project_id"].nunique(),
}

expected_frozen_counts = {
    "files": 10_897,
    "aliquots": 10_703,
    "samples": 10_656,
    "cases": 10_610,
    "projects": 33,
}

if frozen_counts != expected_frozen_counts:
    raise ValueError(
        "Frozen cohort counts differ from the reviewed 104 decision:\n"
        f"Observed: {frozen_counts}\n"
        f"Expected: {expected_frozen_counts}"
    )

frozen_size_gib = (
    frozen_file_index["file_size_bytes"].sum() / (1024**3)
)

frozen_platform_summary = (
    frozen_file_index
    .groupby("platform", as_index=False)
    .agg(
        n_files=("file_id", "nunique"),
        n_aliquots=("aliquot_uuid", "nunique"),
        n_samples=("sample_submitter_id", "nunique"),
        n_cases=("case_uuid", "nunique"),
        total_size_bytes=("file_size_bytes", "sum"),
    )
)

frozen_platform_summary["total_size_gib"] = (
    frozen_platform_summary["total_size_bytes"] / (1024**3)
)

print("HM27 + HM450 cohort selected and validated.")
print(f"Files:          {frozen_counts['files']:,}")
print(f"Aliquots:       {frozen_counts['aliquots']:,}")
print(f"Samples:        {frozen_counts['samples']:,}")
print(f"Cases:          {frozen_counts['cases']:,}")
print(f"Projects:       {frozen_counts['projects']}")
print(f"Download size:  {frozen_size_gib:,.6f} GiB")
print(f"Excluded files: {len(excluded_file_metadata):,} EPICv2")
print()
print(frozen_platform_summary)

HM27 + HM450 cohort selected and validated.
Files:          10,897
Aliquots:       10,703
Samples:        10,656
Cases:          10,610
Projects:       33
Download size:  106.735678 GiB
Excluded files: 53 EPICv2

                         platform  n_files  n_aliquots  n_samples  n_cases  \
0   Illumina Human Methylation 27     2279        2279       2271     2270   
1  Illumina Human Methylation 450     8618        8618       8598     8555   

   total_size_bytes  total_size_gib  
0        1752659375        1.632291  
1      112853901884      105.103386  


In [6]:
# =============================================================================
# Write and validate cohort-freeze metadata
# =============================================================================

source_sha256 = {
    label: calculate_sha256(path)
    for label, path in source_input_paths.items()
}

output_sha256 = {
    label: calculate_sha256(path)
    for label, path in {
        "manifest": FROZEN_MANIFEST_PATH,
        "sample_sheet": FROZEN_SAMPLE_SHEET_PATH,
        "file_index": FROZEN_FILE_INDEX_PATH,
    }.items()
}


# -----------------------------------------------------------------------------
# Platform-specific frozen counts
# -----------------------------------------------------------------------------

platform_counts = []

for row in frozen_platform_summary.itertuples(index=False):
    platform_counts.append(
        {
            "platform": row.platform,
            "n_files": int(row.n_files),
            "n_aliquots": int(row.n_aliquots),
            "n_samples": int(row.n_samples),
            "n_cases": int(row.n_cases),
            "total_file_size_bytes": int(row.total_size_bytes),
            "total_file_size_gib": round(
                float(row.total_size_gib),
                6,
            ),
        }
    )


# -----------------------------------------------------------------------------
# Cohort-freeze record
# -----------------------------------------------------------------------------

cohort_freeze = {
    "schema_version": "1.0",
    "cohort_name": (
        "tcga_primary_tumor_methylation_array_"
        "sesame_beta_values_hm27_hm450"
    ),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "created_by_notebook": (
        "105_tcga_methylation_cohort_freeze.ipynb"
    ),
    "source": {
        "database": "Genomic Data Commons",
        "program": "TCGA",
        "source_export_scope": "all available methylation platforms",
        "raw_inputs": {
            label: {
                "path": to_project_relative_posix_path(
                    path,
                    Paths.root,
                ),
                "sha256": source_sha256[label],
                "size_bytes": int(path.stat().st_size),
            }
            for label, path in source_input_paths.items()
        },
    },
    "gdc_filters": {
        "program": "TCGA",
        "data_category": "DNA Methylation",
        "data_type": "Methylation Beta Value",
        "experimental_strategy": "Methylation Array",
        "workflow_type": "SeSAMe Methylation Beta Estimation",
        "data_format": "TXT",
        "access": "open",
        "tissue_type": "Tumor",
        "tumor_descriptor": "Primary",
    },
    "platform_acquisition_decision": {
        "decision_source_notebook": (
            "104_tcga_methylation_coverage_assessment.ipynb"
        ),
        "acquisition_scope": "source-complete",
        "included_platforms": sorted(INCLUDED_PLATFORMS),
        "excluded_platforms": [PLATFORM_EPICV2],
        "excluded_file_count": int(len(excluded_file_metadata)),
        "rationale": (
            "HM27 is retained because it provides substantial incremental "
            "case coverage in several TCGA projects, including OV and GBM. "
            "HM450 is retained as the principal high-density platform. "
            "EPICv2 is excluded because its 53 files add no exclusive cases "
            "and would introduce a third probe space without increasing "
            "case-level coverage."
        ),
        "deferred_decisions": [
            "one sample or aliquot per case",
            "sample-level methylation quality control",
            "probe-level quality control",
            "cross-platform feature harmonization",
            "cross-platform normalization or correction",
        ],
    },
    "source_cohort_counts": {
        "n_files": int(metadata_file_index["file_id"].nunique()),
        "n_aliquots": int(
            metadata_associations["entity_uuid"].nunique()
        ),
        "n_samples": int(
            source_sample_sheet["Sample ID"].nunique()
        ),
        "n_cases": int(
            metadata_associations["case_uuid"].nunique()
        ),
        "n_projects": int(
            source_sample_sheet["Project ID"].nunique()
        ),
        "total_file_size_bytes": int(
            metadata_file_index["file_size_bytes"].sum()
        ),
        "total_file_size_gib": round(
            metadata_file_index["file_size_bytes"].sum()
            / (1024**3),
            6,
        ),
    },
    "frozen_cohort_counts": {
        "n_files": int(frozen_counts["files"]),
        "n_aliquots": int(frozen_counts["aliquots"]),
        "n_samples": int(frozen_counts["samples"]),
        "n_cases": int(frozen_counts["cases"]),
        "n_projects": int(frozen_counts["projects"]),
        "total_file_size_bytes": int(
            frozen_file_index["file_size_bytes"].sum()
        ),
        "total_file_size_gib": round(
            frozen_size_gib,
            6,
        ),
        "platform_counts": platform_counts,
    },
    "versioned_outputs": {
        "manifest": {
            "path": to_project_relative_posix_path(
                FROZEN_MANIFEST_PATH,
                Paths.root,
            ),
            "sha256": output_sha256["manifest"],
            "n_rows": int(len(reloaded_manifest)),
        },
        "sample_sheet": {
            "path": to_project_relative_posix_path(
                FROZEN_SAMPLE_SHEET_PATH,
                Paths.root,
            ),
            "sha256": output_sha256["sample_sheet"],
            "n_rows": int(len(reloaded_sample_sheet)),
        },
        "file_index": {
            "path": to_project_relative_posix_path(
                FROZEN_FILE_INDEX_PATH,
                Paths.root,
            ),
            "sha256": output_sha256["file_index"],
            "n_rows": int(len(reloaded_file_index)),
        },
        "cohort_freeze": {
            "path": to_project_relative_posix_path(
                FROZEN_COHORT_METADATA_PATH,
                Paths.root,
            ),
        },
    },
    "reproducibility_note": (
        "The source GDC exports are immutable provenance inputs. "
        "The version-controlled HM27/HM450 manifest and file index "
        "define the operational file-level methylation cohort used "
        "by this repository."
    ),
    "scientific_scope": (
        "This artifact freezes the methylation acquisition cohort. "
        "It does not perform sample selection, methylation quality "
        "control, probe filtering, platform harmonization, biological "
        "modeling, causal inference, or therapeutic validation."
    ),
}


# -----------------------------------------------------------------------------
# Write JSON atomically
# -----------------------------------------------------------------------------

temporary_freeze_path = FROZEN_COHORT_METADATA_PATH.with_name(
    FROZEN_COHORT_METADATA_PATH.name + ".tmp"
)

with temporary_freeze_path.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    json.dump(
        cohort_freeze,
        handle,
        indent=2,
        ensure_ascii=False,
    )
    handle.write("\n")

temporary_freeze_path.replace(FROZEN_COHORT_METADATA_PATH)


# -----------------------------------------------------------------------------
# Reload and validate the persisted JSON
# -----------------------------------------------------------------------------

with FROZEN_COHORT_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    reloaded_cohort_freeze = json.load(handle)

if reloaded_cohort_freeze["cohort_name"] != cohort_freeze["cohort_name"]:
    raise ValueError("Persisted cohort-freeze name is incorrect.")

if (
    reloaded_cohort_freeze["frozen_cohort_counts"]["n_files"]
    != expected_frozen_counts["files"]
):
    raise ValueError(
        "Persisted cohort-freeze file count is incorrect."
    )

if set(
    reloaded_cohort_freeze[
        "platform_acquisition_decision"
    ]["included_platforms"]
) != set(INCLUDED_PLATFORMS):
    raise ValueError(
        "Persisted platform-inclusion decision is incorrect."
    )

if (
    reloaded_cohort_freeze[
        "platform_acquisition_decision"
    ]["excluded_platforms"]
    != [PLATFORM_EPICV2]
):
    raise ValueError(
        "Persisted platform-exclusion decision is incorrect."
    )

cohort_freeze_sha256 = calculate_sha256(
    FROZEN_COHORT_METADATA_PATH
)

print("Cohort-freeze metadata written and validated.")
print(
    f"Path:   "
    f"{to_project_relative_posix_path(FROZEN_COHORT_METADATA_PATH, Paths.root)}"
)
print(f"SHA-256: {cohort_freeze_sha256}")
print()
print("Frozen cohort:")
print(f"- Files:     {frozen_counts['files']:,}")
print(f"- Cases:     {frozen_counts['cases']:,}")
print(f"- Platforms: HM27 + HM450")
print(f"- Size:      {frozen_size_gib:,.6f} GiB")

Cohort-freeze metadata written and validated.
Path:   config/manifests/tcga_methylation/gdc_cohort_freeze_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.json
SHA-256: 79124bbdc49253e70aa3aa211775d774306acbdbed1e742c9f4caad5cc0b1a22

Frozen cohort:
- Files:     10,897
- Cases:     10,610
- Platforms: HM27 + HM450
- Size:      106.735678 GiB
